# Customer Segmentation using K-Means Clustering

This notebook is designed for `customer_segments.csv`.

**Dataset columns:** `Customer ID`, `Recency`, `Frequency`, `Monetary`, and an existing `KMeans` label.

The clustering model uses only **Recency, Frequency, and Monetary**. `Customer ID` is an identifier, and the existing `KMeans` column is not used as an input feature to avoid label leakage.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

## 2. Load the CSV file

In [ ]:
# Google Colab: upload customer_segments.csv when prompted.
# If the file is already in the notebook's working directory, it loads directly.
import os

file_path = "customer_segments.csv"

if not os.path.exists(file_path):
    try:
        from google.colab import files
        uploaded = files.upload()
        file_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "Place customer_segments.csv in the same folder as this notebook, then run this cell again."
        )

df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect and clean the data

In [ ]:
print("Columns:", df.columns.tolist())
print("\\nData types:")
print(df.dtypes)
print("\\nMissing values:")
print(df.isnull().sum())

In [ ]:
required = ["Customer ID", "Recency", "Frequency", "Monetary"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=required).copy()
df = df.drop_duplicates(subset=["Customer ID"])
for col in ["Recency", "Frequency", "Monetary"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["Recency", "Frequency", "Monetary"])

# These RFM features should be non-negative for this dataset.
df = df[(df["Recency"] >= 0) & (df["Frequency"] >= 0) & (df["Monetary"] >= 0)].copy()

print("Shape after cleaning:", df.shape)
display(df[required].describe())

## 4. Select features and scale them

In [ ]:
features = ["Recency", "Frequency", "Monetary"]
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature matrix shape:", X_scaled.shape)

## 5. Find a useful number of clusters (Elbow Method)

In [ ]:
inertias = []
k_values = range(2, min(11, len(df)))

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.grid(True, alpha=0.3)
plt.show()

## 6. Fit K-Means

In [ ]:
# Change n_clusters after reviewing the elbow plot if needed.
n_clusters = 4

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df["KMeans_New"] = kmeans.fit_predict(X_scaled)

print("New cluster counts:")
print(df["KMeans_New"].value_counts().sort_index())

if df["KMeans_New"].nunique() > 1 and len(df) > df["KMeans_New"].nunique():
    print("Silhouette score:", round(silhouette_score(X_scaled, df["KMeans_New"]), 3))

## 7. Visualize clusters with PCA

In [ ]:
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(pca_data[:, 0], pca_data[:, 1],
                      c=df["KMeans_New"], cmap="viridis", alpha=0.75)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Customer Segments using K-Means and PCA")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.grid(True, alpha=0.2)
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained:", round(pca.explained_variance_ratio_.sum(), 3))

## 8. Compare average RFM values by cluster

In [ ]:
cluster_summary = df.groupby("KMeans_New")[features].mean().round(2)
cluster_summary["Customer_Count"] = df.groupby("KMeans_New").size()
display(cluster_summary.sort_index())

## 9. Optional: DBSCAN clustering

In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=5)
df["DBSCAN"] = dbscan.fit_predict(X_scaled)

print("DBSCAN cluster labels (-1 means noise):")
print(df["DBSCAN"].value_counts().sort_index())

## 10. Inspect the existing KMeans labels (optional)

In [ ]:
# The CSV already contains a KMeans column. This is the dataset's existing label.
# It was deliberately excluded from model inputs above.
if "KMeans" in df.columns:
    print("Existing labels:")
    print(df["KMeans"].value_counts().sort_index())
    print("\\nExisting labels vs newly generated labels (label numbers need not match):")
    display(pd.crosstab(df["KMeans"], df["KMeans_New"],
                        rownames=["Existing KMeans"], colnames=["New KMeans"]))
else:
    print("No existing KMeans column found.")

## 11. Save the segmented dataset

In [ ]:
output_path = "customer_segments_clustered.csv"
df.to_csv(output_path, index=False)
print("Saved:", output_path)

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("The output file is saved in the current working directory.")